# LLaVA — Image Question Answering

AurumOS reference notebook for vision-language QA with LLaVA.

**Profile-aware model selection** (reads `AURUM_LLAVA_MODEL` from `/etc/aurum/profile.conf`):

| Profile     | Model                              | VRAM (int4) |
|-------------|------------------------------------|-------------|
| lite        | `llava-phi-3-mini` (~2 B params)   | CPU / ~3 GB |
| standard    | `llava-1.5-7b` (int4)              | ~5 GB       |
| pro         | `llava-1.5-13b` (int4)             | ~9 GB       |
| workstation | `llava-v1.6-34b` (int4)            | ~22 GB      |

Before running, download the weights once:

```bash
aurum-cv-download-models llava
```

LLaVA-1.5/1.6 repos are public; LLaVA-NeXT variants may be gated — export `HF_TOKEN` first if so.

In [ ]:
import os

# Resolve the AurumOS symbolic name to a HuggingFace repo id. Same mapping
# as `aurum-cv-download-models llava`.
LLAVA = os.environ.get('AURUM_LLAVA_MODEL', 'llava-7b-int4')
REPO = {
    'llava-phi-int4':  'xtuner/llava-phi-3-mini-hf',
    'llava-7b-int4':   'llava-hf/llava-1.5-7b-hf',
    'llava-13b-int4':  'llava-hf/llava-1.5-13b-hf',
    'llava-34b-int4':  'llava-hf/llava-v1.6-34b-hf',
}.get(LLAVA, LLAVA)
print('Profile model:', LLAVA, '→', REPO)

In [ ]:
import torch
from transformers import AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig

# 4-bit NF4 keeps the 7B model under 6 GB VRAM — comfortable on an 8 GB RTX 5060.
# On CPU-only hosts bitsandbytes prints a warning and falls back to fp32; the
# notebook will still run, just slowly.
quant_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
)

processor = AutoProcessor.from_pretrained(REPO)
model = LlavaForConditionalGeneration.from_pretrained(
    REPO,
    quantization_config=quant_cfg,
    device_map='auto',
)
model.eval()
print('Model loaded on:', model.device)

In [ ]:
from PIL import Image
import urllib.request

# Default sample. Replace with any image path or URL you care about.
SRC = 'https://ultralytics.com/images/bus.jpg'
if SRC.startswith('http'):
    urllib.request.urlretrieve(SRC, '/tmp/aurum-llava-input.jpg')
    SRC = '/tmp/aurum-llava-input.jpg'
image = Image.open(SRC).convert('RGB')
image

In [ ]:
QUESTION = 'Describe this image. How many people do you see and what are they doing?'

# LLaVA-1.5 chat format: USER <image>\nPROMPT\nASSISTANT:
prompt = f'USER: <image>\n{QUESTION}\nASSISTANT:'
inputs = processor(images=image, text=prompt, return_tensors='pt').to(model.device)

with torch.inference_mode():
    output = model.generate(**inputs, max_new_tokens=256, do_sample=False)

answer = processor.batch_decode(output, skip_special_tokens=True)[0]
# The decoded text echoes the prompt; trim back to just the assistant turn.
print(answer.split('ASSISTANT:')[-1].strip())

## Next steps

- Replace `SRC` with your own image (file or URL).
- Replace `QUESTION` with anything — LLaVA handles counting, scene description, OCR-lite, and basic spatial reasoning.
- For batched inference loop over a list of `(image, question)` pairs and call `processor(images=[...], text=[...])`.
- For a richer chat loop, see `02_internvl2_chat.ipynb` (multi-turn) and `03_qwen2_vl.ipynb` (video + image).